In [ ]:
%pip install -q yfinance==0.2.66 requests lxml html5lib
import json, hashlib, platform, sys, warnings
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
# Display figures in Colab
import matplotlib.pyplot as plt
import scipy
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.preprocessing import StandardScaler
import sklearn
import yfinance as yf
from IPython.display import display

ROOT=Path.cwd()
DATA=ROOT/'data'; OUT=ROOT/'outputs'; FIG=OUT/'figures'
for p in [DATA,OUT,FIG]:p.mkdir(parents=True,exist_ok=True)
START='2021-01-01'; END='2025-01-01'; SEED=42
SOURCE='https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'



In [ ]:
def write_json(path, x):path.write_text(json.dumps(x,indent=2,default=str),encoding='utf-8')
def sha(path):return hashlib.sha256(path.read_bytes()).hexdigest()

def image(fig,name):
    fig.tight_layout();fig.savefig(FIG/name,dpi=170,bbox_inches='tight');plt.close(fig)


# Task 1 Data preparation and feature engineering

## 1.1 Ticker selection and data acquisition

In [ ]:
def acquisition():
    constituent_file = DATA / 'constituents.csv'

    if constituent_file.exists():
        cons = pd.read_csv(constituent_file)
    else:
        import requests
        from io import StringIO

        response = requests.get(
            SOURCE,
            headers={"User-Agent": "Mozilla/5.0"},
            timeout=30
        )
        response.raise_for_status()

        cons = pd.read_html(StringIO(response.text))[0]

        needed = ['Symbol', 'Security', 'GICS Sector', 'GICS Sub-Industry']

        if not all(c in cons.columns for c in needed):
            raise RuntimeError('Constituent page schema changed')

        cons = cons[needed].copy()
        cons['retrieved_utc'] = datetime.now(timezone.utc).isoformat()
        cons.to_csv(constituent_file, index=False)

    cons['YahooTicker'] = cons['Symbol'].str.replace('.', '-', regex=False)
    if cons['YahooTicker'].duplicated().any():
        raise ValueError('Duplicate constituent tickers')
    print('Constituent retrieval date:', cons['retrieved_utc'].iloc[0] if 'retrieved_utc' in cons else 'unknown')
    downloaded = {}
    manifest = DATA / 'download_dates.json'
    if manifest.exists():
        downloaded = json.loads(manifest.read_text(encoding='utf-8'))

    bmark = '^GSPC'
    tickers = cons['YahooTicker'].drop_duplicates().tolist()
    all_names = tickers + [bmark]

    missing = []

    for index in range(0, len(all_names), 25):
        batch = all_names[index:index + 25]

        for ticker in batch:
            path = DATA / f'{ticker.replace("^", "INDEX_")}.csv'

            if path.exists():
                continue

            try:
                d = yf.download(
                    ticker,
                    start=START,
                    end=END,
                    interval='1d',
                    auto_adjust=False,
                    actions=True,
                    progress=False,
                    threads=False,
                    multi_level_index=False
                )

                if d is None or d.empty:
                    missing.append(ticker)
                    continue

                if isinstance(d.columns, pd.MultiIndex):
                    d.columns = d.columns.get_level_values(0)

                d.index = pd.to_datetime(d.index).tz_localize(None).normalize()
                d.index.name = 'Date'
                d.to_csv(path)
                downloaded[ticker] = datetime.now(timezone.utc).isoformat()
                write_json(manifest, downloaded)

            except Exception as ex:

                missing.append(ticker)
                print('DOWNLOAD FAILED', ticker, str(ex)[:150])

    write_json(OUT / 'download_failures.json', missing)

    benchmark_file = DATA / 'INDEX_GSPC.csv'

    if not benchmark_file.exists():
        raise RuntimeError(
            'Benchmark unavailable: provide cached benchmark CSV or retry network access'
        )

    return cons


def get_daily(ticker):
    path = DATA / f'{ticker.replace("^", "INDEX_")}.csv'

    if not path.exists():
        return None

    x = pd.read_csv(path, index_col='Date', parse_dates=True)
    x.index = pd.to_datetime(x.index).tz_localize(None).normalize()

    return x.sort_index()


constituents = acquisition()

print(
    "Constituents:", len(constituents),
    "| Cached data files:", len(list(DATA.glob("*.csv")))
)
print("Missing stock files:")

for ticker in constituents['YahooTicker']:
    path = DATA / f'{ticker}.csv'

    if not path.exists():
        print(ticker)

print("Benchmark available:", (DATA / 'INDEX_GSPC.csv').exists())


## 1.2 Data validation and alignment

In [ ]:
benchmark = get_daily('^GSPC')
if benchmark is None or 'Adj Close' not in benchmark:
    raise RuntimeError('Adjusted benchmark data is unavailable')
if benchmark.index.duplicated().any():
    raise ValueError('Duplicate benchmark dates')
benchmark = benchmark.loc[(benchmark.index >= START) & (benchmark.index < END)].copy()
if benchmark.empty or not np.isfinite(benchmark['Adj Close']).all() or (benchmark['Adj Close'] <= 0).any():
    raise ValueError('Invalid benchmark prices')
# Use the benchmark trading calendar throughout, including holiday-shortened weeks.
observation_date = benchmark.index.max()
if benchmark.index.min() != pd.Timestamp('2021-01-04') or observation_date != pd.Timestamp('2024-12-31') or len(benchmark) != 1005:
    raise ValueError('Benchmark does not cover the required 1,005 trading sessions')
corporate_actions = []
valid_daily = {}
exclusions = {}
for _, item in constituents.iterrows():
    ticker = item['YahooTicker']
    daily = get_daily(ticker)
    reasons = []
    if daily is None:
        reasons.append('download unavailable')
    else:
        required = ['Close','Adj Close','Volume']
        if not all(col in daily.columns for col in required):
            reasons.append('missing required fields')
        if daily.index.duplicated().any():
            reasons.append('duplicate dates')
        if not reasons:
            daily = daily.loc[(daily.index >= START) & (daily.index < END)].copy()
            if daily.empty or not np.isfinite(daily[required]).all().all():
                reasons.append('missing price/volume values')
            if len(daily) and ((daily[['Close','Adj Close']] <= 0).any().any() or (daily['Volume'] < 0).any()):
                reasons.append('invalid price or negative volume')
            if len(daily) and (daily['Volume'] == 0).any():
                reasons.append('zero-volume session requiring review')
            if not daily.index.equals(benchmark.index):
                missing = len(benchmark.index.difference(daily.index))
                extra = len(daily.index.difference(benchmark.index))
                reasons.append(f'incomplete or mismatched daily history (missing {missing}, extra {extra})')
            for field in ['Dividends','Stock Splits']:
                if field not in daily.columns:
                    reasons.append(f'missing corporate-action field: {field}')
                elif not np.isfinite(daily[field]).all() or (daily[field] < 0).any():
                    reasons.append(f'invalid corporate-action values: {field}')
            if not reasons:
                close_ratio = daily['Close'].div(daily['Close'].shift(1))
                adjusted_ratio = daily['Adj Close'].div(daily['Adj Close'].shift(1))
                adjustment_factor = daily['Adj Close'].div(daily['Close'])
                for day in daily.index[(daily['Dividends'] != 0) | (daily['Stock Splits'] != 0)]:
                    corporate_actions.append(dict(Ticker=ticker,Date=str(day.date()),
                        Dividend=daily.loc[day,'Dividends'],Split=daily.loc[day,'Stock Splits'],
                        CloseRatio=close_ratio.loc[day],AdjustedRatio=adjusted_ratio.loc[day],
                        AdjustmentFactor=adjustment_factor.loc[day],
                        Volume=daily.loc[day,'Volume'],PriorVolume=daily['Volume'].shift(1).loc[day]))
            if observation_date not in daily.index:
                reasons.append('missing common observation date')
    if reasons:
        exclusions[ticker] = '; '.join(reasons)
    else:
        valid_daily[ticker] = daily
print('Passed daily validation:',len(valid_daily),'| Excluded:',len(exclusions))
# Action-day price ratios help identify possible split discontinuities. They do not independently establish volume adjustment compatibility.
action_checks = pd.DataFrame(corporate_actions,columns=['Ticker','Date','Dividend','Split','CloseRatio','AdjustedRatio','AdjustmentFactor','Volume','PriorVolume'])
action_checks.to_csv(OUT/'corporate_action_checks.csv',index=False)
print('Corporate-action records:',len(action_checks))
print('Split-day ratios outside 0.5 to 1.5:',int(((action_checks['Split'] > 0) & ~action_checks['CloseRatio'].between(.5,1.5) & action_checks['CloseRatio'].notna()).sum()))


## 1.3 Weekly return and weekly beta
### 1.3.1 Mean weekly return

In [ ]:
def valid_weeks(benchmark):
    # Exclude the incomplete opening week and final calendar week, not normal holiday-shortened weeks.
    starts=pd.Timestamp(START);ends=pd.Timestamp(END)-pd.Timedelta(days=1)
    first_fri=starts+pd.offsets.Week(weekday=4)
    # Jan 1 2021 was Friday, and the opening partial week must be excluded.
    if first_fri<starts+pd.Timedelta(days=6):first_fri+=pd.Timedelta(days=7)
    final_fri=ends-pd.Timedelta(days=(ends.weekday()-4)%7)
    return pd.date_range(first_fri,final_fri,freq='W-FRI')

def checked_weekly(d, labels):
    prices=d['Adj Close'].resample('W-FRI').last().reindex(labels)
    # pct_change on a missing label must NOT bridge two weeks
    return prices.div(prices.shift(1)).sub(1).where(prices.notna() & prices.shift(1).notna())

week_labels = valid_weeks(benchmark)
benchmark_returns = checked_weekly(benchmark, week_labels)
if len(week_labels) != 208 or benchmark_returns.notna().sum() != 207:
    raise ValueError('Incomplete benchmark weekly history')
weekly_pairs = {}
for ticker,daily in valid_daily.items():
    returns = checked_weekly(daily, week_labels)
    paired = pd.concat([returns.rename('share'), benchmark_returns.rename('benchmark')],axis=1).dropna()
    if len(paired) != benchmark_returns.notna().sum() or paired['benchmark'].var() <= 0:
        exclusions[ticker] = 'insufficient aligned weekly returns or benchmark variance'
        continue
    weekly_pairs[ticker] = paired
weekly_mean = {ticker: float(pair['share'].mean()) for ticker,pair in weekly_pairs.items()}
print('Shares with aligned returns:',len(weekly_pairs))
print(pd.Series(weekly_mean,name='MeanWeeklyReturn').head())


### 1.3.2 Weekly beta

In [ ]:
weekly_beta = {
    ticker: float(pair['share'].cov(pair['benchmark'])/pair['benchmark'].var())
    for ticker,pair in weekly_pairs.items()
}
weekly_n = {ticker: len(pair) for ticker,pair in weekly_pairs.items()}
print(pd.DataFrame({'MeanWeeklyReturn':weekly_mean,'WeeklyBeta':weekly_beta,'WeeklyN':weekly_n}).head())


## 1.4 Adapted RFM calculations
### 1.4.1 Recency

In [ ]:
rfm_work = {}
for ticker in weekly_pairs:
    history = valid_daily[ticker].loc[:observation_date].copy()
    preceding_20_mean = history['Volume'].shift(1).rolling(20,min_periods=20).mean()
    event_flags = (history['Volume'] > 2*preceding_20_mean) & preceding_20_mean.notna()
    recent_60 = history.tail(60)
    if len(recent_60) != 60 or preceding_20_mean.loc[recent_60.index].isna().any():
        exclusions[ticker] = 'insufficient 20-session baseline or 60-session window'
        continue
    qualifying_dates = history.index[event_flags]
    if len(qualifying_dates) == 0:
        exclusions[ticker] = 'no qualifying high-volume event (R is missing)'
        continue
    last_event = qualifying_dates.max()
    rfm_work[ticker] = dict(history=history,baseline=preceding_20_mean,flags=event_flags,window=recent_60,
                            latest_event=last_event,R=int((observation_date-last_event).days))
print('Shares with valid Recency:',len(rfm_work))
print(pd.Series({t:x['R'] for t,x in rfm_work.items()},name='R').head())


### 1.4.2 Frequency

In [ ]:
for ticker,work in rfm_work.items():
    work['F'] = int(work['flags'].loc[work['window'].index].sum())
print(pd.Series({t:x['F'] for t,x in rfm_work.items()},name='F').head())


### 1.4.3 Monetary

In [ ]:
for ticker,work in rfm_work.items():
    work['M'] = float((work['window']['Close']*work['window']['Volume']).sum())
print(pd.Series({t:x['M'] for t,x in rfm_work.items()},name='M (USD)').head())


## 1.5 Metric interpretation and supporting evidence

In [ ]:
details=constituents.drop_duplicates('YahooTicker').set_index('YahooTicker')
rows=[]; event_rows=[];check_rows=[]
for ticker,work in rfm_work.items():
    h=work['history']; w=work['window']; base=work['baseline'];flags=work['flags']
    rows.append(dict(Ticker=ticker,Security=details.loc[ticker,'Security'],Sector=details.loc[ticker,'GICS Sector'],
        MeanWeeklyReturn=weekly_mean[ticker],WeeklyBeta=weekly_beta[ticker],WeeklyN=weekly_n[ticker],
        R=work['R'],F=work['F'],M=work['M'],LogM=np.log1p(work['M']),
        ObservationDate=str(observation_date.date()),LatestEvent=str(work['latest_event'].date()),
        Last60Start=str(w.index.min().date()),TradingSessions=len(h),
        SplitDays=int(h['Stock Splits'].fillna(0).ne(0).sum()) if 'Stock Splits' in h else np.nan,
        DividendDays=int(h['Dividends'].fillna(0).ne(0).sum()) if 'Dividends' in h else np.nan))
    check_rows.append(dict(Ticker=ticker,WeeklyObservations=weekly_n[ticker],
        MissingWeeks=int(benchmark_returns.notna().sum()-weekly_n[ticker]),DailyRows=len(h),
        SplitDays=rows[-1]['SplitDays'],DividendDays=rows[-1]['DividendDays']))
    for day in w.index:
        event_rows.append(dict(Ticker=ticker,Date=str(day.date()),Volume=h.loc[day,'Volume'],
            Prior20Mean=base.loc[day],Threshold=2*base.loc[day],Qualifies=int(flags.loc[day]),
            Close=h.loc[day,'Close'],TradedValue=h.loc[day,'Close']*h.loc[day,'Volume']))
features=pd.DataFrame(rows)
if len(features)<9: raise RuntimeError('Fewer than 9 valid shares; check data and exclusions')
features.to_csv(OUT/'feature_table.csv',index=False)
pd.DataFrame([{'Ticker':t,'Reason':r} for t,r in exclusions.items()],columns=['Ticker','Reason']).to_csv(OUT/'exclusion_log.csv',index=False)
pd.DataFrame(event_rows).to_csv(OUT/'rfm_daily_evidence.csv',index=False)
pd.DataFrame(check_rows).to_csv(OUT/'validation_log.csv',index=False)
print('Included',len(features),'| Excluded',len(exclusions),'| Observation date',observation_date.date())
display(features[['MeanWeeklyReturn','WeeklyBeta','R','F','M']].describe().T)
worked=features.iloc[0]
print('Worked share:',worked['Ticker'],'Last event:',worked['LatestEvent'],
      'R:',worked['R'],'F:',worked['F'],'M (USD):',f"{worked['M']:,.2f}")
worked_window = pd.DataFrame(event_rows).query('Ticker == @worked.Ticker')
display(worked_window)
work = rfm_work[worked['Ticker']]; day = work['latest_event']; h = work['history']
worked_event = pd.DataFrame([dict(Ticker=worked['Ticker'],Date=str(day.date()),Volume=h.loc[day,'Volume'],
    Prior20Mean=work['baseline'].loc[day],Threshold=2*work['baseline'].loc[day],
    Recency=(observation_date-day).days)])
display(worked_event)
worked_event.to_csv(OUT/'rfm_worked_latest_event.csv',index=False)
worked_window.to_csv(OUT/'rfm_worked_window.csv',index=False)
display(pd.DataFrame([{'Ticker':t,'Reason':r} for t,r in exclusions.items()]))


# Task 2 Stock segmentation and sensitivity analysis

## 2.1 Feature preparation

In [ ]:
from sklearn.decomposition import PCA
if features[['MeanWeeklyReturn','WeeklyBeta','R','F','M']].isna().any().any():
    raise ValueError('Missing clustering inputs')
risk_columns=['MeanWeeklyReturn','WeeklyBeta']; rfm_columns=['R','F','LogM']
risk_raw=features[risk_columns].copy();rfm_raw=features[rfm_columns].copy()
scalers={'risk':StandardScaler(),'rfm':StandardScaler()}
matrices={'risk':scalers['risk'].fit_transform(risk_raw),'rfm':scalers['rfm'].fit_transform(rfm_raw)}
print('Common share count:',len(features))
display(risk_raw.head())
display(rfm_raw.head())
display(features[['MeanWeeklyReturn','WeeklyBeta','R','F','M','LogM']].skew().to_frame('skewness'))
for col in ['MeanWeeklyReturn','WeeklyBeta','R','F','M']:
    q1,q3=features[col].quantile([.25,.75]);iqr=q3-q1
    print(f'{col}: IQR outliers = {((features[col] < q1-1.5*iqr)|(features[col]>q3+1.5*iqr)).sum()}')


## 2.2 Agglomerative clustering

In [ ]:
ward_diagnostics=[];ward_sizes=[];ward_profile_rows=[];chosen_ward={}
for kind,X in matrices.items():
    dendro=linkage(X,method='ward',metric='euclidean')
    fig,ax=plt.subplots(figsize=(11,5))
    dendrogram(dendro,no_labels=True,color_threshold=0,ax=ax)
    ax.set(title=f'Ward dendrogram: {kind.upper()}',xlabel='Shares',ylabel='Linkage distance')
    image(fig,f'{kind}_dendrogram.png')
    display(fig)
    candidates=[]
    for k in range(2,min(8,len(features)-1)+1):
        y=AgglomerativeClustering(n_clusters=k,linkage='ward').fit_predict(X)
        counts=pd.Series(y).value_counts()
        rec=dict(FeatureSet=kind,Method='Ward',K=k,Silhouette=float(silhouette_score(X,y)),
                 Inertia=np.nan,MinClusterSize=int(counts.min()),MaxClusterSize=int(counts.max()))
        ward_diagnostics.append(rec);candidates.append(rec)
        for lab,num in counts.items():
            ward_sizes.append(dict(FeatureSet=kind,Method='Ward',K=k,Cluster=int(lab),Size=int(num)))
            group=features.loc[y==lab]
            rec_profile=dict(FeatureSet=kind,K=k,Cluster=int(lab),Size=int(num))
            rec_profile.update({col:group[col].mean() for col in ['MeanWeeklyReturn','WeeklyBeta','R','F','M']})
            ward_profile_rows.append(rec_profile)
    eligible=[rec for rec in candidates if rec['MinClusterSize']>=2]
    selected=max(eligible or candidates,key=lambda rec:(rec['Silhouette'],-rec['K']))['K']
    chosen_ward[kind]=selected
    features[f'{kind}_ward']=AgglomerativeClustering(n_clusters=selected,linkage='ward').fit_predict(X)
    print(kind,'Ward chosen k =',selected)
ward_diagnostics=pd.DataFrame(ward_diagnostics)
display(ward_diagnostics)
ward_candidate_profiles=pd.DataFrame(ward_profile_rows)
ward_candidate_profiles.to_csv(OUT/'ward_candidate_profiles.csv',index=False)
display(ward_candidate_profiles)


## 2.3 K means clustering

In [ ]:
kmeans_diagnostics=[];kmeans_sizes=[];chosen_kmeans={}
for kind,X in matrices.items():
    candidates=[]
    for k in range(2,min(8,len(features)-1)+1):
        model=KMeans(n_clusters=k,random_state=SEED,n_init=30)
        y=model.fit_predict(X);counts=pd.Series(y).value_counts()
        rec=dict(FeatureSet=kind,Method='KMeans',K=k,Silhouette=float(silhouette_score(X,y)),
                 Inertia=float(model.inertia_),MinClusterSize=int(counts.min()),MaxClusterSize=int(counts.max()))
        candidates.append(rec);kmeans_diagnostics.append(rec)
        for lab,num in counts.items():kmeans_sizes.append(dict(FeatureSet=kind,Method='KMeans',K=k,Cluster=int(lab),Size=int(num)))
    eligible=[rec for rec in candidates if rec['MinClusterSize']>=2]
    selected=max(eligible or candidates,key=lambda rec:(rec['Silhouette'],-rec['K']))['K']
    chosen_kmeans[kind]=selected
    features[f'{kind}_kmeans']=KMeans(n_clusters=selected,random_state=SEED,n_init=30).fit_predict(X)
    print(kind,'KMeans chosen k =',selected)
kmeans_diagnostics=pd.DataFrame(kmeans_diagnostics)
model_selection=pd.concat([ward_diagnostics,kmeans_diagnostics],ignore_index=True)
cluster_sizes=pd.DataFrame(ward_sizes+kmeans_sizes)
model_selection.to_csv(OUT/'model_selection.csv',index=False)
cluster_sizes.to_csv(OUT/'cluster_sizes.csv',index=False)
display(kmeans_diagnostics)
display(cluster_sizes)

## 2.4 Cluster profiles and algorithm comparison

In [ ]:
profile_rows=[]
for kind,X in matrices.items():
    for method in ['ward','kmeans']:
        key=f'{kind}_{method}'
        y=features[key].to_numpy()
        for cluster,group in features.groupby(key):
            rec=dict(FeatureSet=kind,Method='Ward' if method=='ward' else 'KMeans',Cluster=int(cluster),Size=len(group))
            rec.update({col:group[col].mean() for col in ['MeanWeeklyReturn','WeeklyBeta','R','F','M']})
            profile_rows.append(rec)
        if kind=='risk':
            a=features['MeanWeeklyReturn'].to_numpy();b=features['WeeklyBeta'].to_numpy()
            xlabel='Arithmetic mean weekly return';ylabel='Weekly beta'
        else:
            pca=PCA(n_components=2)
            projected=pca.fit_transform(X)
            print(f"{key} PCA explained variance:",pca.explained_variance_ratio_)
            a,b=projected[:,0],projected[:,1]
            xlabel='RFM PCA component 1';ylabel='RFM PCA component 2'
        fig,ax=plt.subplots(figsize=(8,5))
        for lab in sorted(set(y)):
            idx=y==lab;ax.scatter(a[idx],b[idx],s=25,alpha=.8,label=f'Cluster {lab}')
        ax.set(title=f'{kind.upper()} {method.upper()} clusters',xlabel=xlabel,ylabel=ylabel)
        ax.legend();image(fig,f'{key}_scatter.png');display(fig)
cluster_profiles=pd.DataFrame(profile_rows)
cluster_profiles.to_csv(OUT/'cluster_profiles.csv',index=False)
display(cluster_profiles)
algorithm_comparison=[]
for kind in ['risk','rfm']:
    ari=adjusted_rand_score(features[f'{kind}_ward'],features[f'{kind}_kmeans'])
    algorithm_comparison.append(dict(FeatureSet=kind,ARI=ari))
    print(kind,'Ward vs KMeans ARI:',round(ari,3))
pd.DataFrame(algorithm_comparison).to_csv(OUT/'algorithm_comparison.csv',index=False)


## 2.5 RFM sensitivity and correlated inputs
### 2.5.1 Correlation analysis

In [ ]:
rfm_correlations=features[['R','F','LogM']].corr()
rfm_correlations.to_csv(OUT/'rfm_correlations.csv')
display(rfm_correlations)


### 2.5.2 Removal of Monetary

In [ ]:
rf_only_scaled=StandardScaler().fit_transform(features[['R','F']])
features['rfm_ward_without_m']=AgglomerativeClustering(n_clusters=chosen_ward['rfm'],linkage='ward').fit_predict(rf_only_scaled)
monetary_ari=adjusted_rand_score(features['rfm_ward'],features['rfm_ward_without_m'])
rf_only_profiles=features.groupby('rfm_ward_without_m').agg(Count=('Ticker','size'),MeanM=('M','mean'),
    MedianM=('M','median'),MeanR=('R','mean'),MeanF=('F','mean'))
rf_only_profiles.to_csv(OUT/'rf_only_cluster_profiles.csv')
features.to_csv(OUT/'feature_table_with_clusters.csv',index=False)
print('ARI after removing Monetary:',round(monetary_ari,4))
original_profiles=features.groupby('rfm_ward').agg(Count=('Ticker','size'),MeanM=('M','mean'),
    MedianM=('M','median'),MeanR=('R','mean'),MeanF=('F','mean'))
original_profiles.to_csv(OUT/'rfm_ward_traded_value_profiles.csv')
membership_comparison=pd.crosstab(features['rfm_ward'],features['rfm_ward_without_m'])
membership_comparison.to_csv(OUT/'rfm_membership_comparison.csv')
display(membership_comparison)
print('Original RFM profiles:');display(original_profiles)
print('Reduced RF profiles:');display(rf_only_profiles)


In [ ]:
provenance={'timestamp_utc':datetime.now(timezone.utc).isoformat(),'constituent_source':SOURCE,
 'constituent_retrieved_utc':pd.read_csv(DATA/'constituents.csv').get('retrieved_utc',pd.Series(['unknown'])).iloc[0],
 'study_window':[START,END],'observation_date':str(observation_date.date()),
 'provider':'Yahoo Finance via yfinance (auto_adjust=False, actions=True)',
 'download_timestamps':json.loads((DATA/'download_dates.json').read_text()) if (DATA/'download_dates.json').exists() else {},
 'validation':'Exact benchmark trading calendar; complete aligned weekly returns; valid price, volume and action fields',
 'feature_windows':{'weekly':'Friday-ending complete calendar weeks','volume_baseline':20,'R':'full validated history','F_and_M':60},
 'seed':SEED,'kmeans_n_init':30,
 'chosen_k':{'ward':chosen_ward,'kmeans':chosen_kmeans},
 'versions':{'python':sys.version,'pandas':pd.__version__,'numpy':np.__version__,'scipy':scipy.__version__,
             'sklearn':sklearn.__version__,'yfinance':yf.__version__,'matplotlib':matplotlib.__version__},
 'snapshots_sha256':{path.name:sha(path) for path in DATA.glob('*.csv')}}
write_json(OUT/'provenance.json',provenance)
print('Saved outputs/provenance.json')

# Task 3 Business intelligence and critical review

## 3.1 Data mart design

In [ ]:
data_mart_schema = {
 'FactDailyPrices': ['DateKey','ShareKey','Close','AdjClose','Volume','Dividends','StockSplits','SourceRunID'],
 'FactWeeklyReturns': ['WeekKey','ShareKey','BenchmarkKey','WeeklyReturn','BenchmarkWeeklyReturn'],
 'FactFeatureSnapshot': ['ShareKey','ObservationDateKey','CalculationVersion','MeanWeeklyReturn','WeeklyBeta','R','F','M','WeeklyN'],
 'FactClusterAssignments': ['ShareKey','ObservationDateKey','FeatureSet','Algorithm','ModelRunID','SelectedK','Cluster'],
 'DimTime': ['DateKey','CalendarDate','WeekEnding','Year'],
 'DimShare': ['ShareKey','Ticker','Security','SectorKey'],
 'DimSector': ['SectorKey','GICSSector'],
 'DimBenchmark': ['BenchmarkKey','Ticker','Description']
}
display(pd.DataFrame([(name,', '.join(cols)) for name,cols in data_mart_schema.items()],columns=['Table','Columns']))


### 3.2.1 Cluster profiles and sector composition

In [ ]:
for key in ['risk_ward','risk_kmeans','rfm_ward','rfm_kmeans']:
    sector_composition=pd.crosstab(features['Sector'],features[key])
    sector_composition.to_csv(OUT/f'{key}_sector_composition.csv')
    fig,ax=plt.subplots(figsize=(10,5))
    sector_composition.plot(kind='bar',stacked=True,ax=ax)
    ax.set(xlabel='Sector',ylabel='Share count',title=f'Sector composition: {key.replace("_", " ").upper()}')
    image(fig,f'{key}_sector_composition.png');display(fig)
    display(sector_composition)